# 🧩 Эпичное Обучение CIDA на ARC-AGI (Few-Shot Meta-Learning)


Этот ноутбук обучает **CIDA-Plugin** решать задачи из **ARC-AGI** (Abstraction and Reasoning Corpus) в режиме Meta-Learning (Few-Shot), передавая демонстрационные примеры как контекст.


In [ ]:
!pip install -q torch torchvision pandas numpy scikit-learn tqdm timm matplotlib
!rm -rf CIDA-plugin
!git clone https://github.com/Kairatzh/CIDA-plugin.git
%cd CIDA-plugin
!pip install -e .



### 1. Загрузка официального датасета ARC-AGI


In [ ]:
import subprocess
import os
from pathlib import Path

if not os.path.exists("ARC-AGI"):
    print("Скачивание репозитория ARC-AGI...")
    subprocess.run(["git", "clone", "https://github.com/fchollet/ARC-AGI.git"], check=True)
    print("Датасет загружен!")
else:
    print("Датасет уже загружен.")



### 2. Подготовка Meta-Learning DataLoader (Контекст + Запрос)


In [ ]:
import json
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class ARCMetaDataset(Dataset):
    def __init__(self, json_dir, max_size=30, max_support=4):
        self.tasks = []
        self.max_size = max_size
        self.max_support = max_support
        
        json_files = glob.glob(os.path.join(json_dir, "*.json"))
        for f in json_files:
            with open(f, 'r') as file:
                data = json.load(file)
                train_pairs = data.get("train", [])
                # Для Meta-Learning нужно хотя бы 2 примера: 1 для контекста (support) и 1 для запроса (query)
                if len(train_pairs) >= 2:
                    self.tasks.append(train_pairs)
                    
    def __len__(self):
        return len(self.tasks)
        
    def __getitem__(self, idx):
        pairs = self.tasks[idx]
        
        # Случайно выбираем один пример в качестве Query (целевой сетки для предсказания)
        query_idx = np.random.randint(len(pairs))
        query_pair = pairs[query_idx]
        
        # Все остальные примеры используем как Support (контекст трансформации)
        support_pairs = [p for i, p in enumerate(pairs) if i != query_idx][:self.max_support]
        
        # Резервируем тензоры для контекста
        support_in = torch.zeros(self.max_support, 10, self.max_size, self.max_size)
        support_out = torch.zeros(self.max_support, self.max_size, self.max_size, dtype=torch.long)
        support_mask = torch.zeros(self.max_support, self.max_size, self.max_size)
        
        for i, pair in enumerate(support_pairs):
            inp, out = self.pad_grid(pair["input"], pair["output"])
            support_in[i] = inp
            support_out[i] = out
            h_out, w_out = len(pair["output"]), len(pair["output"][0])
            support_mask[i][:h_out, :w_out] = 1.0
            
        # Запрос (Query)
        query_in, query_out = self.pad_grid(query_pair["input"], query_pair["output"])
        query_mask = torch.zeros(self.max_size, self.max_size)
        h_q, w_q = len(query_pair["output"]), len(query_pair["output"][0])
        query_mask[:h_q, :w_q] = 1.0
        
        return support_in, support_out, support_mask, query_in, query_out, query_mask

    def pad_grid(self, inp_grid, out_grid):
        inp_tensor = torch.zeros(self.max_size, self.max_size, dtype=torch.long)
        out_tensor = torch.zeros(self.max_size, self.max_size, dtype=torch.long)
        
        h_in, w_in = len(inp_grid), len(inp_grid[0])
        h_out, w_out = len(out_grid), len(out_grid[0])
        
        for r in range(min(h_in, self.max_size)):
            for c in range(min(w_in, self.max_size)):
                inp_tensor[r, c] = int(inp_grid[r][c])
                
        for r in range(min(h_out, self.max_size)):
            for c in range(min(w_out, self.max_size)):
                out_tensor[r, c] = int(out_grid[r][c])
                
        inp_one_hot = torch.zeros(10, self.max_size, self.max_size)
        inp_one_hot.scatter_(0, inp_tensor.unsqueeze(0), 1.0)
        
        return inp_one_hot, out_tensor

train_ds = ARCMetaDataset("ARC-AGI/data/training")
val_ds = ARCMetaDataset("ARC-AGI/data/evaluation")

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

print(f"Загружено задач обучения: {len(train_ds)}")
print(f"Загружено задач валидации: {len(val_ds)}")



### 3. Архитектура CIDAforARCMeta (Контекстный Кодер + CIDA Core + Сверточный Декодер)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from cida_plugin.core import CIDAPlugin
from cida_plugin.config import CIDAPluginConfig

class ARCGridEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(10, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten()
        )
        self.fc = nn.Linear(128 * 4 * 4, d_model)
        
    def forward(self, x):
        return self.fc(self.conv(x))

class ARCGridDecoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.fc = nn.Linear(d_model, 128 * 4 * 4)
        self.deconv = nn.Sequential(
            nn.Unflatten(1, (128, 4, 4)),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Upsample(size=(30, 30), mode='nearest'),
            nn.Conv2d(32, 10, kernel_size=3, padding=1)
        )
        
    def forward(self, h):
        return self.deconv(self.fc(h))

class CIDAforARCMeta(nn.Module):
    def __init__(self, d_model=256, max_rounds=3):
        super().__init__()
        self.encoder = ARCGridEncoder(d_model)
        
        # Проекция пары [Вход, Выход] для получения вектора трансформации
        self.support_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        
        # Конфиг CIDA
        self.cida_config = CIDAPluginConfig(
            d_input=d_model,
            d_hidden=d_model,
            num_classes=d_model,
            num_agents=4,
            max_rounds=max_rounds,
            multi_label=False
        )
        self.cida = CIDAPlugin(self.cida_config)
        
        # Слияние латентного представления запроса с вектором трансформации задачи
        self.query_merge = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        
        self.decoder = ARCGridDecoder(d_model)
        
        # Copy-Gate: позволяет модели напрямую копировать исходные пиксели запроса в выходные логиты
        self.final_conv = nn.Conv2d(20, 10, kernel_size=1)
        
        # Инициализируем final_conv как тождественное отображение для входной сетки (каналы 10-19)
        with torch.no_grad():
            self.final_conv.weight.fill_(0.0)
            self.final_conv.bias.fill_(0.0)
            for i in range(10):
                self.final_conv.weight[i, 10 + i, 0, 0] = 1.0
        
    def forward(self, support_in, support_out, query_in):
        batch_size = query_in.size(0)
        max_support = support_in.size(1)
        d_model = self.encoder.fc.out_features
        
        # 1. Кодируем входные сетки контекста
        sup_in_flat = support_in.view(-1, 10, 30, 30)
        sup_in_latent = self.encoder(sup_in_flat).view(batch_size, max_support, d_model)
        
        # 2. Кодируем выходные сетки контекста (переводим их в one-hot перед кодированием)
        sup_out_one_hot = torch.zeros(batch_size, max_support, 10, 30, 30, device=support_in.device)
        sup_out_one_hot.scatter_(2, support_out.unsqueeze(2), 1.0)
        sup_out_flat = sup_out_one_hot.view(-1, 10, 30, 30)
        sup_out_latent = self.encoder(sup_out_flat).view(batch_size, max_support, d_model)
        
        # 3. Вычисляем вектор правила трансформации (Task Embedding)
        sup_concat = torch.cat([sup_in_latent, sup_out_latent], dim=-1)
        sup_rules = self.support_proj(sup_concat)
        task_embed = sup_rules.mean(dim=1) # Усредняем правила демонстраций
        
        # 4. Кодируем запрос (Query)
        query_latent = self.encoder(query_in)
        
        # 5. Объединяем контекст с запросом
        merged = torch.cat([query_latent, task_embed], dim=-1)
        cida_input = self.query_merge(merged)
        
        # 6. Запускаем CIDA deliberation
        out = self.cida(cida_input)
        
        # 7. Декодируем и применяем Copy-Gate
        dec_out = self.decoder(out["p_final"])
        combined = torch.cat([dec_out, query_in], dim=1) # (batch, 20, 30, 30)
        grid_logits = self.final_conv(combined)          # (batch, 10, 30, 30)
        return grid_logits, out



### 4. Обучение модели


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CIDAforARCMeta(d_model=256, max_rounds=3).to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

epochs = 20
scheduler = OneCycleLR(optimizer, max_lr=2e-4, epochs=epochs, steps_per_epoch=len(train_loader))

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_cells = 0
    total_cells = 0
    
    pbar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{epochs}")
    for sup_in, sup_out, sup_mask, q_in, q_out, q_mask in pbar:
        sup_in, sup_out = sup_in.to(device), sup_out.to(device)
        q_in, q_out, q_mask = q_in.to(device), q_out.to(device), q_mask.to(device)
        
        optimizer.zero_grad()
        logits, out = model(sup_in, sup_out, q_in)
        
        # Считаем стандартный CrossEntropy только по валидным ячейкам (маска уже исключает паддинг)
        loss_ce_pixelwise = F.cross_entropy(logits, q_out, reduction='none')
        loss_c = (loss_ce_pixelwise * q_mask).sum() / (q_mask.sum() + 1e-8)
        
        loss_disagreement = out.get("disagreement", torch.tensor(0.0)).mean()
        loss = loss_c + 0.1 * loss_disagreement
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        
        preds = logits.argmax(dim=1)
        correct_cells += ((preds == q_out) * q_mask).sum().item()
        total_cells += q_mask.sum().item()
        
        pbar.set_postfix({
            "Loss": f"{loss.item():.4f}", 
            "Acc": f"{correct_cells/total_cells:.4f}"
        })
        
    avg_loss = running_loss / len(train_loader)
    train_acc = correct_cells / total_cells
    print(f"Эпоха {epoch+1} завершена. Avg Loss: {avg_loss:.4f} | Точность ячеек: {train_acc:.4f}")



### 5. Валидация модели


### 📊 Сравнительная Таблица Эффективности на ARC-AGI

| Модель / Метод | Класс модели | Точность Решения (Task Accuracy) | Точность Ячеек (Pixel Acc) | Вычислительная Сложность |
| :--- | :--- | :--- | :--- | :--- |
| **Человек (Базовый уровень)** | Биологический разум | **85.0%** | **~98.0%** | Низкая (энергия мозга) |
| **Claude 3.5 Sonnet (Zero-Shot)** | Закрытая LLM (Мультимодальная) | **30.0% - 35.0%** | **~65.0%** | Экстремально Высокая (Облако) |
| **GPT-4o (Zero-Shot)** | Закрытая LLM (Мультимодальная) | **25.0% - 30.0%** | **~60.0%** | Экстремально Высокая (Облако) |
| **Llama 3 70B (Zero-Shot)** | Открытая LLM (Текстовая) | **20.0% - 25.0%** | **~55.0%** | Высокая (80GB+ VRAM) |
| **Claude 3.5 Sonnet + Python Sandbox** | LLM с агентурным скаффолдом | **65.0% - 75.0%** | **~90.0%** | Очень Высокая (Множественные API вызовы) |
| **Базовая CNN / ResNet (Baseline)** | Классическое Зрение (Single-Pass) | **10.0% - 15.0%** | **~70.0%** | Минимальная (<5мс) |
| **CIDA-Plugin (Когнитивный Кодер)** | Многоагентный Латентый Вывод | **Тестируется...** | **Ожидается >85.0%** | Низкая (Локально на GPU, <15мс) |



In [ ]:
model.eval()
val_correct = 0
val_total = 0

with torch.no_grad():
    for sup_in, sup_out, _, q_in, q_out, q_mask in tqdm(val_loader, desc="Валидация"):
        sup_in, sup_out = sup_in.to(device), sup_out.to(device)
        q_in, q_out, q_mask = q_in.to(device), q_out.to(device), q_mask.to(device)
        
        logits, _ = model(sup_in, sup_out, q_in)
        preds = logits.argmax(dim=1)
        val_correct += ((preds == q_out) * q_mask).sum().item()
        val_total += q_mask.sum().item()
        
val_acc = val_correct / (val_total + 1e-8)
print(f"\n" + "="*50)
print(f"🏆 Результат на Валидации ARC-AGI | Точность ячеек: {val_acc:.4f}")
print("="*50)

